# 05_distilbert_finetuning

## Course connection

This follows the Hugging Face sequence-classification workflow studied in `references/course/Transformers_classification.ipynb`. Unlike the previous notebook, all DistilBERT layers can adapt to BANKING77.

## Fine-tuned DistilBERT — E3

**What this does:** allows every DistilBERT layer to update on BANKING77.

**Why:** this tests whether adapting contextual representations to banking-specific intent boundaries improves over the frozen encoder.


In [1]:
# E3 uses the same data split and evaluation rules as E2 so the only major difference is encoder training.
# The pretrained classification head is replaced with a new 77-intent head.
# E3 allows the full encoder to adapt to the banking intent boundaries.
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import json
from src.data.load_data import load_banking77
from src.models.transformer import run_transformer_experiment
dataset = load_banking77(validation_size=0.1, seed=42)
result = run_transformer_experiment(dataset, frozen=False, learning_rate=2e-05, epochs=3, max_length=64, seed=42)
print({"history": result["history"], "training_seconds": result["training_seconds"], "trainable_parameters": result["trainable_parameters"], "total_parameters": result["total_parameters"]})
Path("results/logs").mkdir(parents=True, exist_ok=True)
Path("results/logs/05_distilbert_finetuning_history.json").write_text(json.dumps({"history": result["history"], "training_seconds": result["training_seconds"], "trainable_parameters": result["trainable_parameters"], "total_parameters": result["total_parameters"]}, indent=2))
result["model"].save_pretrained("results/05_distilbert_finetuning_checkpoint")
result["tokenizer"].save_pretrained("results/05_distilbert_finetuning_checkpoint")

Map:   0%|          | 0/1001 [00:00<?, ? examples/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertForSequenceClassification LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
classifier.bias         | MISSING    | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[W927 16:53:33.113418940 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 20971520 bytes (free: 21430272, total: 8184725504).
[W927 16:53:34.259811605 CUDACachingAllocator.cpp:3934] memory allocation failed with OOM on device 0 while trying to allocate 2097152 bytes (free: 15794176, total: 8184725504).


{'history': [{'epoch': 1, 'train_loss': 3.014057704989737, 'validation_loss': 1.4942719215875144, 'validation_accuracy': 0.7492507492507493, 'validation_macro_f1': 0.6907611097246606}, {'epoch': 2, 'train_loss': 0.9808533036081135, 'validation_loss': 0.5681735232814804, 'validation_accuracy': 0.8781218781218781, 'validation_macro_f1': 0.8615489222292838}, {'epoch': 3, 'train_loss': 0.4092900436657107, 'validation_loss': 0.4020388238466941, 'validation_accuracy': 0.8941058941058941, 'validation_macro_f1': 0.8912902368931706}], 'training_seconds': 200.94349573100044, 'trainable_parameters': 67012685, 'total_parameters': 67012685}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('results/05_distilbert_finetuning_checkpoint/tokenizer_config.json',
 'results/05_distilbert_finetuning_checkpoint/tokenizer.json')